Evaluate the pre-trained weights of OpenCLIP on the test data of RSICD.

In [1]:
import open_clip
open_clip.list_pretrained()

/Users/matheus/miniforge3/envs/liu-general/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[('RN50', 'openai'),
 ('RN50', 'yfcc15m'),
 ('RN50', 'cc12m'),
 ('RN101', 'openai'),
 ('RN101', 'yfcc15m'),
 ('RN50x4', 'openai'),
 ('RN50x16', 'openai'),
 ('RN50x64', 'openai'),
 ('ViT-B-32', 'openai'),
 ('ViT-B-32', 'laion400m_e31'),
 ('ViT-B-32', 'laion400m_e32'),
 ('ViT-B-32', 'laion2b_e16'),
 ('ViT-B-32', 'laion2b_s34b_b79k'),
 ('ViT-B-32', 'datacomp_xl_s13b_b90k'),
 ('ViT-B-32', 'datacomp_m_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_clip_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_laion_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_image_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_text_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_basic_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_s128m_b4k'),
 ('ViT-B-32', 'datacomp_s_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_clip_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_laion_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_image_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_text_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_basic_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_s13m_b4k'),
 ('ViT-

Load the train/validation data.

In [3]:
# load the RSICD test dataset
import pandas as pd
dataset = pd.read_json("../data/dataset_rsicd.json")

In [5]:
dataset['filename'] = dataset['images'].apply(lambda x: x['filename'])
dataset['imgid'] = dataset['images'].apply(lambda x: x['imgid'])
dataset['sentences'] = dataset['images'].apply(lambda x: x['sentences'])
dataset['split'] = dataset['images'].apply(lambda x: x['split'])
dataset['sentids'] = dataset['images'].apply(lambda x: x['sentids'])
dataset['raw_captions'] = dataset['sentences'].apply(lambda x: [sentence['raw'] for sentence in x])

train_df = dataset[dataset['split'] == 'train']
val_df   = dataset[dataset['split'] == 'val']
test_df  = dataset[dataset['split'] == 'test']

print("Number of training images:", len(train_df))
print("Number of validation images:", len(val_df))
print("Number of test images:", len(test_df))

dataset.head(3)

Number of training images: 8734
Number of validation images: 1094
Number of test images: 1093


,images,dataset,filename,imgid,sentences,split,sentids,raw_captions
0,"{'filename': 'airport_1.jpg', 'imgid': 0, 'sen...",RSICD,airport_1.jpg,0,"[{'tokens': ['many', 'planes', 'are', 'parked'...",train,"[0, 1, 2, 3, 4]",[many planes are parked next to a long buildin...
1,"{'filename': 'airport_10.jpg', 'imgid': 1, 'se...",RSICD,airport_10.jpg,1,"[{'tokens': ['some', 'planes', 'are', 'parked'...",train,"[5, 6, 7, 8, 9]","[some planes are parked in an airport ., the a..."
2,"{'filename': 'airport_100.jpg', 'imgid': 2, 's...",RSICD,airport_100.jpg,2,"[{'tokens': ['many', 'planes', 'are', 'parked'...",train,"[10, 11, 12, 13, 14]",[many planes are parked in an airport near man...


Now we have the captions for each image -- we now need the class labels and also the images...

In [ ]:
import torch
from PIL import Image

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer('ViT-B-32')

In [ ]:
image = preprocess(Image.open("docs/CLIP.png")).unsqueeze(0)
text = tokenizer(["a diagram", "a dog", "a cat"])

with torch.no_grad(), torch.autocast("cuda"):
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print("Label probs:", text_probs)  # prints: [[1., 0., 0.]]